# TeCo paper-style RAW FRPs — CPU prepare
Select **CPU** and **Run all**. Wait for CPU PREPARATION COMPLETE, then open the GPU notebook.

**Scope:** Task 1 including control reading; all recoverable stored word fixations, including rereadings. Positive/negative agent labels must agree lexically and in word context; neutral is excluded here. Whole-sentence labels need not agree. No full blink/saccade/keypress timeline is available, so this is a **raw-FRP adaptation**, not a regression-cleaned replication. No Figure 3 or missing eye-control signal is invented.

**Storage:** at most **2 GB additional**, including about **0.61 GB** temporary permutation maps plus safety reserve. Approximately **1.24 GB** remains for accepted individual float32 EEG trials and metadata. All 126 channels and 201 samples (125 Hz, −600…1000 ms) are retained. Temporary permutation files are deleted only after successful results; individual trials remain. No new copy of the large native pickle is saved. Reader checkpoints are verified and skipped on resume. An interrupted unfinished reader may need repeating. Do not run both notebooks simultaneously.

Native inputs are read from `Thesis/Data/teco_pickles/<reader>_with_total_task1/task1.pickle` (known alternative layouts are also checked). Private labels are read from `Thesis/gpt uploaded files/word_sentiment_eeg/teco/`. Persian text and labels stay on Drive. Code remains on GitHub.


## Connect Drive, check quota and load pinned public code
No GitHub token required. Mount the account containing the Thesis folder.

In [ ]:
# Runtime check: run before installation, Drive access, or analysis.
import os
import subprocess

REQUIRED_RUNTIME = 'CPU'
if os.environ.get('COLAB_TPU_ADDR') or os.environ.get('TPU_NAME'):
    raise RuntimeError(f'This notebook needs {REQUIRED_RUNTIME}, not TPU. Choose Runtime > Change runtime type, then rerun all.')
try:
    _gpu_probe = subprocess.run(
        ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=15, check=False)
except FileNotFoundError:
    _gpu_names = ''
except subprocess.TimeoutExpired:
    raise RuntimeError('GPU detection timed out. Reconnect the runtime and rerun this cell.') from None
else:
    if _gpu_probe.returncode:
        raise RuntimeError('GPU detection failed. Reconnect the runtime and rerun this cell.')
    _gpu_names = _gpu_probe.stdout.strip()
if REQUIRED_RUNTIME == 'GPU' and not _gpu_names:
    raise RuntimeError('GPU required. Choose Runtime > Change runtime type > T4 GPU, then Run all. No setup or analysis has started.')
if REQUIRED_RUNTIME == 'CPU' and _gpu_names:
    raise RuntimeError('CPU notebook: choose Runtime > Change runtime type > CPU (or hardware accelerator None), then Run all to avoid using GPU quota.')
print('Runtime verified:', _gpu_names if _gpu_names else 'CPU', flush=True)


In [ ]:
from google.colab import drive, auth
from pathlib import Path
import os, sys, subprocess, json
from IPython.display import display, Markdown, Image
import pandas as pd

drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
OUT=THESIS/'Results/word_sentiment_eeg/teco_paper_raw_staged'
REPO=Path('/content/word-sentiment-eeg-teco-paper')
CODE_REVISION='9651d15759091b01635180282962d9e8129c9a3b'
def git(*args):
    result=subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=',*map(str,args)],
                          text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                          env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
    if result.returncode:
        raise RuntimeError('Git setup failed:\n'+result.stdout)
    if result.stdout:print(result.stdout,end='',flush=True)
if not REPO.exists():
    git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary checkout contains edits. Preserve them before updating.')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    git('-C',REPO,'fetch','--depth','1','origin',CODE_REVISION)
    fetched=subprocess.check_output(['git','-C',str(REPO),'rev-parse','FETCH_HEAD^{commit}'],text=True).strip()
    if fetched!=CODE_REVISION:raise RuntimeError('Fetched code differs from the pinned commit.')
    git('-C',REPO,'checkout','--detach','FETCH_HEAD')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    raise RuntimeError('Code checkout verification failed.')
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements-paper.txt')],check=True)
print('Code revision:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

# A mounted Drive filesystem disk_usage value can reflect the runtime volume;
# use the authenticated Drive quota instead.
auth.authenticate_user()
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials,_=google.auth.default()
response=AuthorizedSession(credentials).get('https://www.googleapis.com/drive/v3/about',
    params={'fields':'storageQuota,user(emailAddress)'},timeout=30)
response.raise_for_status();quota=response.json()['storageQuota']
print('Quota account:',response.json().get('user',{}).get('emailAddress','unavailable'))
if 'limit' not in quota:
    raise RuntimeError('Drive did not expose a finite storage quota; verify free space before starting.')
FREE_DRIVE_BYTES=max(0,int(quota['limit'])-int(quota.get('usage',0)))
print(f'Account free space: {FREE_DRIVE_BYTES/1e9:.2f} GB. Workflow checks the additional space needed before source reads.')
print('Destination:',OUT)


## CPU: prepare

In [ ]:
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[sys.executable,'-u',str(REPO/'scripts/teco_paper_stages.py'),'--thesis-root',str(THESIS),'--phase','prepare', '--free-drive-bytes',str(FREE_DRIVE_BYTES)]
run_with_progress(command,OUT/'status/prepare.json',activity='TeCo paper prepare: reader checkpoints and 2 GB cap')


## Next: GPU analysis
After CPU PREPARATION COMPLETE, open [TeCo GPU analysis](https://colab.research.google.com/github/parmisbathayan/word-sentiment-eeg/blob/main/notebooks/teco_paper_analyze_gpu.ipynb?authuser=2). It loads the saved trials; it does not repeat EEG extraction.